In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Iniciar la sesión de Spark
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("IntroPySpark_Finanzas") \
    .getOrCreate()

In [3]:
import pandas as pd
import numpy as np

print("Generando archivo CSV sintético de 50,000 transacciones (esto tomará unos segundos)...")
n_rows = 50000
np.random.seed(42) # Para reproducibilidad en clase

# Distribución realista de tipos de transacción
tipos = ["PAYMENT", "TRANSFER", "CASH_OUT", "CASH_IN", "DEBIT"]
probabilidades = [0.35, 0.15, 0.30, 0.18, 0.02]

# Construimos un DataFrame de Pandas con datos aleatorios pero coherentes
df_pd = pd.DataFrame({
    "step": np.random.randint(1, 100, n_rows),
    "type": np.random.choice(tipos, n_rows, p=probabilidades),
    "amount": np.round(np.random.exponential(scale=100000, size=n_rows), 2),
    "nameOrig": ["C" + str(i) for i in np.random.randint(10000, 99999, n_rows)],
    "oldbalanceOrg": np.round(np.random.exponential(scale=150000, size=n_rows), 2),
    "nameDest": ["C" + str(i) for i in np.random.randint(10000, 99999, n_rows)],
    "oldbalanceDest": np.round(np.random.exponential(scale=200000, size=n_rows), 2),
})

# Lógica básica de alteración de saldos
df_pd["newbalanceOrig"] = np.maximum(df_pd["oldbalanceOrg"] - df_pd["amount"], 0)
df_pd["newbalanceDest"] = df_pd["oldbalanceDest"] + df_pd["amount"]

# Inyectar Fraude (Imbalanceado): ~1% de fraude, concentrado en transferencias grandes
df_pd["isFraud"] = 0
mask_fraude = (df_pd["type"].isin(["TRANSFER", "CASH_OUT"])) & (df_pd["amount"] > 150000) & (np.random.rand(n_rows) < 0.05)
df_pd.loc[mask_fraude, "isFraud"] = 1

# Guardamos el dataset en el disco local de Colab
df_pd.to_csv("paysim_sample.csv", index=False)
print("¡Archivo 'paysim_sample.csv' creado con éxito!")

Generando archivo CSV sintético de 50,000 transacciones (esto tomará unos segundos)...
¡Archivo 'paysim_sample.csv' creado con éxito!


In [4]:
# 3. LECTURA DEL DATASET CON PYSPARK
# Veamos cómo usar spark.read para cargar un dataset en un DataFrame de PySpark
print("\nCargando datos con PySpark SQL...")

df = spark.read.csv(
    "paysim_sample.csv",
    header=True,       # La primera fila tiene los nombres de las columnas
    inferSchema=True   # PySpark adivina el tipo de dato (entero, string, float)
)

print("Esquema inferido por Spark:")
df.printSchema()

print("Primeras 5 filas distribuidas:")
df.show(5)


Cargando datos con PySpark SQL...
Esquema inferido por Spark:
root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)

Primeras 5 filas distribuidas:
+----+-------+---------+--------+-------------+--------+--------------+------------------+--------------+-------+
|step|   type|   amount|nameOrig|oldbalanceOrg|nameDest|oldbalanceDest|    newbalanceOrig|newbalanceDest|isFraud|
+----+-------+---------+--------+-------------+--------+--------------+------------------+--------------+-------+
|  52|PAYMENT| 75260.18|  C22062|    302563.46|  C47999|     159891.14|227303.28000000003|     235151.32|      0|
|  93|PAYMENT|123158.12|  C264

In [5]:
# 4. TRANSFORMACIONES (Evaluación Perezosa en acción)
# Supongamos que queremos analizar solo transacciones grandes (> 5000)
df_filtered = df.filter(df.amount > 5000) \
                .select("type", "amount", "oldbalanceOrg", "isFraud")

# En este punto, no se ha ejecutado nada. Solo se ha definido el plan.
print("Conteo de transacciones grandes:", df_filtered.count()) # Aquí se dispara la acción

Conteo de transacciones grandes: 47535


In [6]:
# 5. AGREGACIONES FINANCIERAS
# ¿Cuál es la cantidad/monto promedio por tipo de transacción?
df.groupBy("type") \
    .agg(F.avg("amount").alias("monto_promedio"),
         F.count("amount").alias("total_operaciones")) \
    .orderBy(F.desc("total_operaciones")) \
    .show()

# 6. FEATURE ENGINEERING (Creación de columnas)
# Diferencia de balance en origen
df = df.withColumn("errorBalanceOrig", (df.oldbalanceOrg - df.amount) - df.newbalanceOrig)
df.select("amount", "oldbalanceOrg", "newbalanceOrig", "errorBalanceOrig").show(5)

+--------+------------------+-----------------+
|    type|    monto_promedio|total_operaciones|
+--------+------------------+-----------------+
| PAYMENT|100323.95260991524|            17468|
|CASH_OUT| 99110.33413713805|            14788|
| CASH_IN|100349.04294904118|             9125|
|TRANSFER| 98591.54850900407|             7552|
|   DEBIT| 99888.70061855673|             1067|
+--------+------------------+-----------------+

+---------+-------------+------------------+------------------+
|   amount|oldbalanceOrg|    newbalanceOrig|  errorBalanceOrig|
+---------+-------------+------------------+------------------+
| 75260.18|    302563.46|227303.28000000003|               0.0|
|123158.12|     54678.29|               0.0|-68479.82999999999|
| 90241.22|    119847.21|29605.990000000005|               0.0|
|258352.55|     10966.08|               0.0|        -247386.47|
|230348.78|     37060.15|               0.0|        -193288.63|
+---------+-------------+------------------+-----------

**EJERCICIO PROPUESTO**: Cargar y realizar un procesamiento sencillo (2-3 pasos) sobre al dataset que contiene datos sociodemográficos y características de viviendas en distritos de California.

https://github.com/gakudo-ai/open-datasets/blob/main/housing.csv